# Phase 04B — Traditional Regression Baselines

Initialization and frozen-input preflight only. No model is instantiated, fitted, tuned, or evaluated in this notebook.

In [1]:
from pathlib import Path
import hashlib, json, platform, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import scipy, sklearn, joblib, threadpoolctl
from sklearn.model_selection import GroupKFold

PROJECT_ROOT = Path(r'E:\hdc-vr-pilot')
PHASE_DIR = PROJECT_ROOT / 'experiments' / 'phase_04b_traditional_regression_baselines'
PRIMARY_PATH = PROJECT_ROOT / 'experiments' / 'phase_03_multimodal_dataset_labeling' / 'data' / 'primary_without_performance.csv'
FOLD_PATH = PROJECT_ROOT / 'experiments' / 'phase_03_multimodal_dataset_labeling' / 'data' / 'fold_assignments.csv'
EXPECTED_SHA256 = 'e4dc943af21851bace49345f6336f9c88f82613ca3a26b3c433efe7dfb041f6f'
NON_FEATURE_COLUMNS = ['subject_id', 'session_id', 'run_id', 'difficulty_level_raw', 'difficulty_level', 'run_key', 'target_class', 'target_score', 'outer_fold']
SHARED_COLUMNS = NON_FEATURE_COLUMNS
assert PROJECT_ROOT.is_dir() and PRIMARY_PATH.is_file() and FOLD_PATH.is_file()
actual_sha256 = hashlib.sha256(FOLD_PATH.read_bytes()).hexdigest()
assert actual_sha256 == EXPECTED_SHA256, f'Frozen fold checksum mismatch: {actual_sha256}'
print('FROZEN FOLD CHECKSUM: PASS')
print('Phase 03 inputs loaded read-only from absolute paths.')

FROZEN FOLD CHECKSUM: PASS
Phase 03 inputs loaded read-only from absolute paths.


In [2]:
primary = pd.read_csv(PRIMARY_PATH)
folds = pd.read_csv(FOLD_PATH)
failures = []
def check(condition, label):
    if not bool(condition):
        failures.append(label)
    return bool(condition)

rows = len(primary); subjects = primary['subject_id'].nunique(); total_columns = len(primary.columns)
non_feature_present = [c for c in NON_FEATURE_COLUMNS if c in primary.columns]
predictive_features = total_columns - len(NON_FEATURE_COLUMNS)
target_values = sorted(primary['target_score'].dropna().unique().tolist())
target_missing = int(primary['target_score'].isna().sum())
unique_run_keys = int(primary['run_key'].nunique(dropna=True))
check(rows == 419, 'modeling rows != 419'); check(subjects == 35, 'subjects != 35'); check(total_columns == 1185, 'total columns != 1185')
check(len(NON_FEATURE_COLUMNS) == 9 and non_feature_present == NON_FEATURE_COLUMNS, 'specified non-feature columns invalid')
check(predictive_features == 1176, 'primary predictive features != 1176')
check(target_values == [1.0, 2.0, 3.0, 4.0], 'target_score values invalid'); check(target_missing == 0, 'target_score has missing values')
check(primary['run_key'].notna().all() and unique_run_keys == 419, 'primary run_key invalid')
check(len(folds) == 419, 'fold assignment rows != 419')
fold_unique_run_keys = int(folds['run_key'].nunique(dropna=True))
check(folds['run_key'].notna().all() and fold_unique_run_keys == 419, 'fold run_key invalid')
primary_duplicates = int(primary['run_key'].duplicated().sum()); fold_duplicates = int(folds['run_key'].duplicated().sum())
primary_keys, fold_keys = set(primary['run_key']), set(folds['run_key'])
missing_from_folds = len(primary_keys - fold_keys); extra_in_folds = len(fold_keys - primary_keys)
alignment_pass = check(missing_from_folds == 0 and extra_in_folds == 0 and primary_duplicates == 0 and fold_duplicates == 0, 'run_key alignment failure')
joined = primary[SHARED_COLUMNS].merge(folds[SHARED_COLUMNS], on='run_key', how='inner', validate='one_to_one', suffixes=('_primary', '_fold'))
shared_mismatches = {}
for column in SHARED_COLUMNS:
    if column != 'run_key':
        count = int((joined[f'{column}_primary'] != joined[f'{column}_fold']).sum())
        shared_mismatches[column] = count
shared_pass = check(len(joined) == 419 and all(v == 0 for v in shared_mismatches.values()), 'shared fields inconsistent')
outer_folds = int(folds['outer_fold'].nunique(dropna=True))
check(folds['outer_fold'].notna().all() and outer_folds == 5, 'outer folds invalid')
print(f'Rows={rows}; subjects={subjects}; columns={total_columns}; primary features={predictive_features}')
print(f'Run-key alignment={"PASS" if alignment_pass else "FAIL"}; shared-field consistency={"PASS" if shared_pass else "FAIL"}')

Rows=419; subjects=35; columns=1185; primary features=1176
Run-key alignment=PASS; shared-field consistency=PASS


In [3]:
outer_isolation = {}; inner_feasibility = {}; inner_pass = True
for fold_value in sorted(folds['outer_fold'].unique()):
    test_rows = folds[folds['outer_fold'] == fold_value]
    train_rows = folds[folds['outer_fold'] != fold_value]
    overlap = sorted(set(train_rows['subject_id']) & set(test_rows['subject_id']))
    outer_isolation[str(fold_value)] = {'train_subjects': int(train_rows['subject_id'].nunique()), 'test_subjects': int(test_rows['subject_id'].nunique()), 'subject_overlap': overlap, 'pass': len(overlap) == 0}
    unique_train_subjects = int(train_rows['subject_id'].nunique())
    splits = []; fold_inner_pass = unique_train_subjects >= 3
    if fold_inner_pass:
        for inner_index, (train_idx, val_idx) in enumerate(GroupKFold(n_splits=3).split(train_rows, groups=train_rows['subject_id']), start=1):
            train_subjects = set(train_rows.iloc[train_idx]['subject_id']); val_subjects = set(train_rows.iloc[val_idx]['subject_id'])
            split_overlap = sorted(train_subjects & val_subjects)
            split_pass = len(split_overlap) == 0
            fold_inner_pass = fold_inner_pass and split_pass
            splits.append({'inner_fold': inner_index, 'train_subjects': len(train_subjects), 'validation_subjects': len(val_subjects), 'subject_overlap': split_overlap, 'pass': split_pass})
    inner_feasibility[str(fold_value)] = {'outer_training_unique_subjects': unique_train_subjects, 'generated_inner_splits': len(splits), 'splits': splits, 'pass': fold_inner_pass}
    inner_pass = inner_pass and fold_inner_pass
outer_pass = check(all(x['pass'] for x in outer_isolation.values()), 'outer subject isolation failure')
inner_pass = check(inner_pass, 'inner GroupKFold feasibility failure')
print(f'Outer subject isolation: {"PASS" if outer_pass else "FAIL"}')
print(f'Inner 3-fold GroupKFold feasibility: {"PASS" if inner_pass else "FAIL"}')

Outer subject isolation: PASS
Inner 3-fold GroupKFold feasibility: PASS


## Dummy Regressor baselines

The following executed cells evaluate only frozen-outer-fold mean and median baselines.

In [4]:
from pathlib import Path
import hashlib
import json
import os
from datetime import datetime, timezone

from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

EXPECTED_SHA256 = 'e4dc943af21851bace49345f6336f9c88f82613ca3a26b3c433efe7dfb041f6f'
AUDIT_PATH = PHASE_DIR / 'audits' / 'phase04b_input_and_fold_audit.json'
CONTRACT_PATH = PHASE_DIR / 'configs' / 'phase04b_experiment_contract.json'
current_sha256 = hashlib.sha256(FOLD_PATH.read_bytes()).hexdigest()
current_audit = json.loads(AUDIT_PATH.read_text(encoding='utf-8'))
current_contract = json.loads(CONTRACT_PATH.read_text(encoding='utf-8'))
prerequisites = {
    'checksum': current_sha256 == EXPECTED_SHA256,
    'input_audit': current_audit.get('overall_pass') is True or current_audit.get('overall_pass_before_notebook_persistence') is True,
    'rows': len(primary) == 419,
    'subjects': primary['subject_id'].nunique() == 35,
    'predictive_features': current_audit.get('predictive_feature_count') == 1176,
    'unique_run_key': primary['run_key'].nunique() == 419,
    'outer_folds': folds['outer_fold'].nunique() == 5,
    'outer_subject_isolation': all(item['pass'] for item in current_audit['outer_subject_isolation'].values()),
    'inner_groupkfold_feasibility': all(item['pass'] for item in current_audit['inner_groupkfold_feasibility'].values()),
    'contract_sha': current_contract.get('phase03_frozen_fold_sha256') == EXPECTED_SHA256,
}
assert all(prerequisites.values()), f'DUMMY REGRESSOR EXECUTION: BLOCKED: {prerequisites}'
assert (primary['target_score'] == primary['difficulty_level'].astype(float)).all()
print('DUMMY REGRESSOR EXECUTION PRECHECK: PASS')
print(f'FROZEN FOLD SHA-256: {current_sha256}')


DUMMY REGRESSOR EXECUTION PRECHECK: PASS
FROZEN FOLD SHA-256: e4dc943af21851bace49345f6336f9c88f82613ca3a26b3c433efe7dfb041f6f


In [5]:
import time
import warnings

RESULT_DIR = PHASE_DIR / 'results'
PREDICTIONS_DIR = RESULT_DIR / 'predictions'
FOLD_METRICS_DIR = RESULT_DIR / 'fold_metrics'
CHECKPOINTS_DIR = RESULT_DIR / 'checkpoints'
SUMMARIES_DIR = RESULT_DIR / 'summaries'

def atomic_csv(frame, path):
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)

def atomic_json(payload, path):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + chr(10), encoding='utf-8')
    temporary.replace(path)

def safe_spearman(y_true, y_pred):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        value = spearmanr(y_true, y_pred).statistic
    return float(value) if np.isfinite(value) else np.nan

model_specs = [
    ('Dummy Regressor Mean', 'dummy_mean', 'mean'),
    ('Dummy Regressor Median', 'dummy_median', 'median'),
]
completed = {}
audit_models = {}
for model_name, model_slug, strategy in model_specs:
    model_oof = []
    fold_records = []
    model_checkpoint_dir = CHECKPOINTS_DIR / model_slug
    model_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    audit_folds = {}
    for fold_value in sorted(folds['outer_fold'].unique()):
        test_mask = folds['outer_fold'].eq(fold_value).to_numpy()
        train_rows = primary.loc[~test_mask].copy()
        test_rows = primary.loc[test_mask].copy()
        train_subjects = set(train_rows['subject_id'])
        test_subjects = set(test_rows['subject_id'])
        subject_overlap = train_subjects & test_subjects
        assert not subject_overlap, f'subject leakage in outer fold {fold_value}'
        x_train = np.zeros((len(train_rows), 1), dtype=float)
        x_test = np.zeros((len(test_rows), 1), dtype=float)
        y_train = train_rows['target_score'].to_numpy(dtype=float)
        y_test = test_rows['target_score'].to_numpy(dtype=float)
        estimator = DummyRegressor(strategy=strategy)
        fit_start = time.perf_counter()
        estimator.fit(x_train, y_train)
        fit_time = time.perf_counter() - fit_start
        prediction_start = time.perf_counter()
        raw_prediction = estimator.predict(x_test)
        prediction_time = time.perf_counter() - prediction_start
        bounded_prediction = np.clip(raw_prediction, 1.0, 4.0)
        expected_statistic = float(np.mean(y_train) if strategy == 'mean' else np.median(y_train))
        assert np.allclose(raw_prediction, expected_statistic, rtol=0.0, atol=1e-12)
        checkpoint_prediction = test_rows[['run_key', 'subject_id', 'session_id', 'run_id', 'outer_fold', 'target_score']].copy()
        checkpoint_prediction.insert(0, 'strategy', strategy)
        checkpoint_prediction.insert(0, 'model_slug', model_slug)
        checkpoint_prediction.insert(0, 'model', model_name)
        checkpoint_prediction['prediction_raw'] = raw_prediction
        checkpoint_prediction['prediction_bounded'] = bounded_prediction
        checkpoint_prediction['absolute_error_raw'] = np.abs(y_test - raw_prediction)
        checkpoint_prediction['absolute_error_bounded'] = np.abs(y_test - bounded_prediction)
        fold_record = {
            'model': model_name, 'model_slug': model_slug, 'strategy': strategy, 'outer_fold': int(fold_value),
            'train_rows': int(len(train_rows)), 'test_rows': int(len(test_rows)),
            'train_subjects': int(len(train_subjects)), 'test_subjects': int(len(test_subjects)),
            'subject_overlap_count': int(len(subject_overlap)), 'train_target_mean': float(np.mean(y_train)),
            'train_target_median': float(np.median(y_train)), 'prediction_value': expected_statistic,
            'mae_raw': float(mean_absolute_error(y_test, raw_prediction)), 'mae_bounded': float(mean_absolute_error(y_test, bounded_prediction)),
            'rmse_raw': float(np.sqrt(mean_squared_error(y_test, raw_prediction))), 'rmse_bounded': float(np.sqrt(mean_squared_error(y_test, bounded_prediction))),
            'r2_raw': float(r2_score(y_test, raw_prediction)), 'r2_bounded': float(r2_score(y_test, bounded_prediction)),
            'spearman_raw': safe_spearman(y_test, raw_prediction), 'spearman_bounded': safe_spearman(y_test, bounded_prediction),
            'fit_time_seconds': fit_time, 'prediction_time_seconds': prediction_time,
        }
        atomic_csv(checkpoint_prediction, model_checkpoint_dir / f'{model_slug}_fold_{fold_value}_predictions.csv')
        atomic_json(fold_record, model_checkpoint_dir / f'{model_slug}_fold_{fold_value}_metrics.json')
        model_oof.append(checkpoint_prediction)
        fold_records.append(fold_record)
        audit_folds[str(fold_value)] = {
            'train_test_subject_overlap_count': int(len(subject_overlap)),
            'train_target_mean': float(np.mean(y_train)), 'train_target_median': float(np.median(y_train)),
            'prediction_value': expected_statistic,
            'prediction_equals_training_statistic': bool(np.allclose(raw_prediction, expected_statistic, rtol=0.0, atol=1e-12)),
            'outer_test_target_leakage_detected': False,
        }
    oof = pd.concat(model_oof, ignore_index=True).sort_values('run_key').reset_index(drop=True)
    assert len(oof) == 419 and oof['run_key'].nunique() == 419 and not oof['run_key'].duplicated().any()
    assert oof['prediction_raw'].notna().all() and oof['prediction_bounded'].notna().all()
    assert oof['outer_fold'].nunique() == 5 and oof['prediction_bounded'].between(1.0, 4.0).all()
    for fold_value, group in oof.groupby('outer_fold'):
        assert group['prediction_raw'].nunique() == 1, f'non-constant predictions in fold {fold_value}'
    fold_metrics = pd.DataFrame(fold_records).sort_values('outer_fold').reset_index(drop=True)
    atomic_csv(oof, PREDICTIONS_DIR / f'{model_slug}_oof.csv')
    atomic_csv(fold_metrics, FOLD_METRICS_DIR / f'{model_slug}_fold_metrics.csv')
    y_all = oof['target_score'].to_numpy(dtype=float)
    raw_all = oof['prediction_raw'].to_numpy(dtype=float)
    bounded_all = oof['prediction_bounded'].to_numpy(dtype=float)
    summary = {
        'model': model_name, 'model_slug': model_slug, 'strategy': strategy,
        'oof_rows': int(len(oof)), 'oof_unique_run_keys': int(oof['run_key'].nunique()),
        'oof_mae_raw': float(mean_absolute_error(y_all, raw_all)), 'oof_mae_bounded': float(mean_absolute_error(y_all, bounded_all)),
        'oof_rmse_raw': float(np.sqrt(mean_squared_error(y_all, raw_all))), 'oof_rmse_bounded': float(np.sqrt(mean_squared_error(y_all, bounded_all))),
        'oof_r2_raw': float(r2_score(y_all, raw_all)), 'oof_r2_bounded': float(r2_score(y_all, bounded_all)),
        'oof_spearman_raw': safe_spearman(y_all, raw_all), 'oof_spearman_bounded': safe_spearman(y_all, bounded_all),
        'fold_mae_bounded_mean': float(fold_metrics['mae_bounded'].mean()), 'fold_mae_bounded_std': float(fold_metrics['mae_bounded'].std(ddof=1)),
        'total_fit_time_seconds': float(fold_metrics['fit_time_seconds'].sum()), 'total_prediction_time_seconds': float(fold_metrics['prediction_time_seconds'].sum()), 'status': 'COMPLETE',
    }
    completed[model_slug] = {'oof': oof, 'fold_metrics': fold_metrics, 'summary': summary}
    audit_models[model_slug] = {
        'oof_rows': int(len(oof)), 'unique_run_keys': int(oof['run_key'].nunique()),
        'missing_run_keys': int(len(set(primary['run_key']) - set(oof['run_key']))), 'extra_run_keys': int(len(set(oof['run_key']) - set(primary['run_key']))),
        'duplicate_run_keys': int(oof['run_key'].duplicated().sum()), 'missing_prediction_raw': int(oof['prediction_raw'].isna().sum()),
        'missing_prediction_bounded': int(oof['prediction_bounded'].isna().sum()),
        'bounded_prediction_min': float(oof['prediction_bounded'].min()), 'bounded_prediction_max': float(oof['prediction_bounded'].max()),
        'fold_coverage': sorted(int(x) for x in oof['outer_fold'].unique()), 'fold_checks': audit_folds,
    }
print('Both Dummy Regressor outer-fold evaluations completed.')


Both Dummy Regressor outer-fold evaluations completed.


In [6]:
summary_frame = pd.DataFrame([completed['dummy_mean']['summary'], completed['dummy_median']['summary']])
per_level_records = []
for model_slug, result in completed.items():
    oof = result['oof']
    for target_score, group in oof.groupby('target_score', sort=True):
        per_level_records.append({
            'model': group['model'].iloc[0], 'model_slug': model_slug, 'target_score': float(target_score), 'n_samples': int(len(group)),
            'mae_raw': float(group['absolute_error_raw'].mean()), 'mae_bounded': float(group['absolute_error_bounded'].mean()),
            'mean_prediction_raw': float(group['prediction_raw'].mean()), 'mean_prediction_bounded': float(group['prediction_bounded'].mean()),
        })
atomic_csv(summary_frame, SUMMARIES_DIR / 'dummy_regressor_summary.csv')
atomic_csv(pd.DataFrame(per_level_records).sort_values(['model_slug', 'target_score']), SUMMARIES_DIR / 'dummy_regressor_per_level_mae.csv')
dummy_config = {
    'models': [{'name': 'Dummy Regressor Mean', 'slug': 'dummy_mean', 'strategy': 'mean'}, {'name': 'Dummy Regressor Median', 'slug': 'dummy_median', 'strategy': 'median'}],
    'frozen_fold_sha256': current_sha256, 'primary_data_path': str(PRIMARY_PATH), 'fold_assignments_path': str(FOLD_PATH),
    'target_definition': 'target_score = difficulty_level; values = 1.0, 2.0, 3.0, 4.0', 'target_interpretation': 'bounded difficulty-induced workload proxy regression',
    'primary_metric': 'MAE', 'bounded_prediction_rule': 'prediction_bounded = np.clip(prediction_raw, 1.0, 4.0); no rounding before primary evaluation',
    'oof_calculation_rule': 'Concatenate complete frozen outer-test predictions and compute OOF metrics directly over all 419 rows.',
    'scikit_learn_version': sklearn.__version__, 'python_executable': sys.executable, 'utc_timestamp': datetime.now(timezone.utc).isoformat(),
    'random_seed': 'NOT_APPLICABLE', 'inner_cv': 'NOT_REQUIRED_FOR_DUMMY',
}
atomic_json(dummy_config, PHASE_DIR / 'configs' / 'dummy_regressor_configuration.json')
search_path = PHASE_DIR / 'configs' / 'regression_model_search_space.json'
search_space = json.loads(search_path.read_text(encoding='utf-8'))
for entry in search_space['models']:
    if entry['name'] in {'Dummy Regressor mean', 'Dummy Regressor median'}:
        entry['status'] = 'COMPLETE'
search_space['status'] = 'DUMMY_BASELINES_COMPLETE / REMAINING_MODELS_NOT_STARTED'
atomic_json(search_space, search_path)
audit_pass = True
for audit_model in audit_models.values():
    audit_pass = audit_pass and audit_model['oof_rows'] == 419 and audit_model['unique_run_keys'] == 419
    audit_pass = audit_pass and audit_model['missing_run_keys'] == 0 and audit_model['extra_run_keys'] == 0 and audit_model['duplicate_run_keys'] == 0
    audit_pass = audit_pass and audit_model['missing_prediction_raw'] == 0 and audit_model['missing_prediction_bounded'] == 0
    audit_pass = audit_pass and audit_model['bounded_prediction_min'] >= 1.0 and audit_model['bounded_prediction_max'] <= 4.0
    audit_pass = audit_pass and all(item['train_test_subject_overlap_count'] == 0 and item['prediction_equals_training_statistic'] and not item['outer_test_target_leakage_detected'] for item in audit_model['fold_checks'].values())
dummy_audit = {'frozen_fold_sha256': current_sha256, 'models': audit_models, 'overall_pass': bool(audit_pass), 'utc_timestamp': datetime.now(timezone.utc).isoformat()}
atomic_json(dummy_audit, PHASE_DIR / 'audits' / 'dummy_regressor_oof_coverage_audit.json')
print('Saved:', PREDICTIONS_DIR / 'dummy_mean_oof.csv')
print('Saved:', PREDICTIONS_DIR / 'dummy_median_oof.csv')
print('Saved:', SUMMARIES_DIR / 'dummy_regressor_summary.csv')
print('Saved:', PHASE_DIR / 'audits' / 'dummy_regressor_oof_coverage_audit.json')


Saved: E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\results\predictions\dummy_mean_oof.csv
Saved: E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\results\predictions\dummy_median_oof.csv
Saved: E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\results\summaries\dummy_regressor_summary.csv
Saved: E:\hdc-vr-pilot\experiments\phase_04b_traditional_regression_baselines\audits\dummy_regressor_oof_coverage_audit.json


In [7]:
print('FINAL DUMMY STATUS: BOTH COMPLETE')
print('READY FOR RIDGE: PENDING NOTEBOOK PERSISTENCE AUDIT')
display(summary_frame)


FINAL DUMMY STATUS: BOTH COMPLETE
READY FOR RIDGE: PENDING NOTEBOOK PERSISTENCE AUDIT


,model,model_slug,strategy,oof_rows,oof_unique_run_keys,oof_mae_raw,oof_mae_bounded,oof_rmse_raw,oof_rmse_bounded,oof_r2_raw,oof_r2_bounded,oof_spearman_raw,oof_spearman_bounded,fold_mae_bounded_mean,fold_mae_bounded_std,total_fit_time_seconds,total_prediction_time_seconds,status
0,Dummy Regressor Mean,dummy_mean,mean,419,419,0.998814,0.998814,1.116974,1.116974,-0.000015,-0.000015,-0.005086,-0.005086,0.998831,0.00655,0.001940,0.000138,COMPLETE
1,Dummy Regressor Median,dummy_median,median,419,419,0.998807,0.998807,1.204358,1.204358,-0.162603,-0.162603,-0.003221,-0.003221,0.998824,0.00654,0.002104,0.000136,COMPLETE
